In [29]:
import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings('ignore')

### 개요
- section6만 따로 때낸 데이터로 머신러닝을 시작한다.
- 추출한 데이터는 이용 시간과 이용 거리가 최소 5이상
- 시작이나 종료 스테이션이 ST-1035, ST-454, ST-471인 경우
- 시간을 시와 분으로 나눠서 정리함

In [30]:
df = pd.read_csv('../../Data/Zero/section6.csv')
# df.drop('Unnamed: 0',axis=1,inplace=True)
# df = df.loc[:,['기준_날짜','시','분','집계_기준','시작_대여소_ID','종료_대여소_ID', '전체_건수','전체_이용_분','전체_이용_거리']]
df

,기준_날짜,집계_기준,시작_대여소_ID,시작_대여소명,종료_대여소_ID,종료_대여소명,전체_건수,전체_이용_분,전체_이용_거리,기준_시간,시,분
0,20240101,출발시간,ST-454,대조동_024_1,ST-1331,응암3동_002_1,1.0,115.0,11170.0,5.0,0,5
1,20240101,출발시간,ST-454,대조동_024_1,ST-1036,역촌동_001_1,1.0,5.0,860.0,15.0,0,15
2,20240101,출발시간,ST-1035,불광2동_021_1,ST-3058,진관동_038_8,1.0,14.0,1821.0,20.0,0,20
3,20240101,출발시간,ST-454,대조동_024_1,ST-349,북가좌1동_008_1,1.0,26.0,4473.0,20.0,0,20
4,20240101,출발시간,ST-454,대조동_024_1,ST-1025,응암1동_005_1,1.0,12.0,1960.0,25.0,0,25
...,...,...,...,...,...,...,...,...,...,...,...,...
86154,20241231,출발시간,ST-454,대조동_020_1,ST-1036,역촌동_044_1,1.0,5.0,860.0,2335.0,23,35
86155,20241231,출발시간,ST-454,대조동_020_1,ST-455,응암1동_023_1,2.0,38.0,4182.0,2335.0,23,35
86156,20241231,출발시간,ST-454,대조동_020_1,ST-459,녹번동_043_1,1.0,10.0,1490.0,2340.0,23,40
86157,20241231,도착시간,ST-3169,구산동_060_1,ST-454,대조동_020_1,1.0,8.0,3030.0,2340.0,23,40


In [39]:
df1 = pd.read_csv('../../Data/은평구_스테이션_군집화_1차_자전거댓수_추가.csv')
df1['시작_대여소_ID'] = df1['대여소_ID']
addr = df1['대여소_ID'].unique().tolist()
station = ['ST-1035','ST-454','ST-471']
df1

,대여소_ID,주소1,주소2,위도,경도,1분기_전체건수,2분기_전체건수,3분기_전체건수,4분기_전체건수,연간_전체건수,cluster_12_custom,LCD,QR,시작_대여소_ID
0,ST-453,진관동 86-31,구파발역 2번출구,37.636234,126.918999,8392,19134,14804,10860,53190,1,11.0,10.0,ST-453
1,ST-1483,진관2로 15-46,은평구 진관2로 15-46,37.639259,126.918907,3696,8120,6612,4986,23414,1,10.0,0.0,ST-1483
2,ST-1481,진관2로 지하 15-25,구파발역 환승센터,37.638252,126.919456,3112,6034,4838,4348,18332,1,15.0,0.0,ST-1481
3,ST-1329,진관3로 77,은평뉴타운구파발9단지,37.642609,126.921478,2696,5372,4268,3298,15634,1,10.0,0.0,ST-1329
4,ST-2244,진관4로 26 진관고등학교,NaN,37.643509,126.924103,2082,5474,4326,3502,15384,1,0.0,10.0,ST-2244
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,ST-462,신사동 352-2,NaN,37.590809,126.913689,10730,23348,15650,14046,63774,12,NaN,NaN,ST-462
94,ST-463,증산동 199-8,증산역 4번출구,37.584381,126.909897,7233,12914,9790,8834,38771,12,10.0,0.0,ST-463
95,ST-2257,가좌로 255,NaN,37.591812,126.914833,6633,13360,8368,7994,36355,12,0.0,10.0,ST-2257
96,ST-1034,증산동 239,디지털미디어 시티역 4번출구,37.577202,126.902351,3796,5736,4514,4626,18672,12,10.0,0.0,ST-1034


In [38]:
# 1) capacity 만들기
df1['LCD'] = pd.to_numeric(df1['LCD'], errors='coerce').fillna(0)
df1['QR'] = pd.to_numeric(df1['QR'], errors='coerce').fillna(0)
df1['capacity'] = df1['LCD'] + df1['QR']

# 2) 시간대 만들기
# df['기준_시간'] = df['기준_시간'].fillna(0).astype(int)
# df['hour'] = (df['기준_시간'] // 100).astype(int)

# 3) 대여/반납 집계
# 집계_기준이 '출발시간'/'도착시간' 있는 경우
start_df = df[df['집계_기준'].astype(str).str.contains('출발', na=False)]
end_df   = df[df['집계_기준'].astype(str).str.contains('도착', na=False)]

start_cnt = (start_df.groupby(['기준_날짜','시','시작_대여소_ID'])['전체_건수']
                     .sum()
                     .reset_index(name='대여건수'))

end_cnt = (end_df.groupby(['기준_날짜','시','종료_대여소_ID'])['전체_건수']
                   .sum()
                   .reset_index(name='반납건수')
                   .rename(columns={'종료_대여소_ID':'시작_대여소_ID'}))

# 4) 순유출입 = 대여 - 반납
flow = pd.merge(start_cnt, end_cnt,
                on=['기준_날짜','시','시작_대여소_ID'], how='outer').fillna(0)
flow['순유출입'] = flow['대여건수'] - flow['반납건수']

# 5) capacity 붙이기
flow = flow.merge(df1[['시작_대여소_ID','capacity']],
                  on='시작_대여소_ID', how='left')

# 6) risk 지표 & 가능/불가능 라벨
# (순유출입 / capacity) 높을수록 고갈 위험 ↑
flow['risk'] = flow['순유출입'] / flow['capacity'].replace(0, np.nan)
flow['risk'] = flow['risk'].fillna(0)

# 임계치: 필요하면 조정 (예: 0.3)
THRESH = 0.3
flow['가능'] = (flow['risk'] < THRESH).astype(int)

flow.head()


,기준_날짜,시,시작_대여소_ID,대여건수,반납건수,순유출입,capacity,risk,가능
0,20240101,0,ST-1035,1.0,0.0,1.0,6.0,0.166667,1
1,20240101,0,ST-454,4.0,3.0,1.0,10.0,0.100000,1
2,20240101,1,ST-1035,1.0,0.0,1.0,6.0,0.166667,1
3,20240101,1,ST-454,3.0,0.0,3.0,10.0,0.300000,0
4,20240101,1,ST-471,0.0,1.0,-1.0,10.0,-0.100000,1


In [33]:
# test용 데이터에도 동일한 전처리 적용
def build_flow(df, stations_df, thresh=0.3):
    df = df.copy()

    # 2025_data.parquet에는 기준_시간대만 있음 -> 시, 분 생성
    if '시' not in df.columns:
        df['기준_시간대'] = pd.to_numeric(df['기준_시간대'], errors='coerce').fillna(0).astype(int)
        df['시'] = (df['기준_시간대'] // 100).astype(int)
        df['분'] = (df['기준_시간대'] % 100).astype(int)

    start_df = df[df['집계_기준'].astype(str).str.contains('출발', na=False)]
    end_df   = df[df['집계_기준'].astype(str).str.contains('도착', na=False)]

    start_cnt = (start_df.groupby(['기준_날짜','시','시작_대여소_ID'])['전체_건수']
                         .sum().reset_index(name='대여건수'))
    end_cnt = (end_df.groupby(['기준_날짜','시','종료_대여소_ID'])['전체_건수']
                       .sum().reset_index(name='반납건수')
                       .rename(columns={'종료_대여소_ID':'시작_대여소_ID'}))

    flow = pd.merge(start_cnt, end_cnt,
                    on=['기준_날짜','시','시작_대여소_ID'], how='outer').fillna(0)
    flow['순유출입'] = flow['대여건수'] - flow['반납건수']

    flow = flow.merge(stations_df[['시작_대여소_ID','capacity']],
                      on='시작_대여소_ID', how='left')
    flow['capacity'] = flow['capacity'].fillna(0)

    flow['risk'] = flow['순유출입'] / flow['capacity'].replace(0, np.nan)
    flow['risk'] = flow['risk'].fillna(0)
    flow['가능'] = (flow['risk'] < thresh).astype(int)

    return flow

# test flow 생성
test_raw = pd.read_parquet('../../Data/2025_data.parquet')
flow_test = build_flow(test_raw, df1, thresh=THRESH)

feature_cols = ['capacity', 'risk', '순유출입']
X_test = flow_test[feature_cols]
y_test = flow_test['가능']
flow_test.head()


,기준_날짜,시,시작_대여소_ID,대여건수,반납건수,순유출입,capacity,risk,가능
0,20250109,0,ST-1029,0.0,2.0,-2.0,10.0,-0.200000,1
1,20250109,0,ST-1032,1.0,1.0,0.0,26.0,0.000000,1
2,20250109,0,ST-1038,1.0,0.0,1.0,10.0,0.100000,1
3,20250109,0,ST-1331,1.0,0.0,1.0,9.0,0.111111,1
4,20250109,0,ST-1481,1.0,0.0,1.0,15.0,0.066667,1


In [34]:
flow.info()

<class 'pandas.DataFrame'>
RangeIndex: 19255 entries, 0 to 19254
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   기준_날짜      19255 non-null  int64  
 1   시          19255 non-null  int64  
 2   시작_대여소_ID  19255 non-null  str    
 3   대여건수       19255 non-null  float64
 4   반납건수       19255 non-null  float64
 5   순유출입       19255 non-null  float64
 6   capacity   19255 non-null  float64
 7   risk       19255 non-null  float64
 8   가능         19255 non-null  int64  
dtypes: float64(5), int64(3), str(1)
memory usage: 1.4 MB


====================================================
====================================================

In [35]:
flow.columns

Index(['기준_날짜', '시', '시작_대여소_ID', '대여건수', '반납건수', '순유출입', 'capacity', 'risk',
       '가능'],
      dtype='str')

In [36]:
# test용 데이터에도 동일한 전처리 적용
def build_flow(df, stations_df, thresh=0.3):
    df = df.copy()

    # 2025_data.parquet에는 기준_시간대만 있음 -> 시, 분 생성
    if '시' not in df.columns:
        df['기준_시간대'] = pd.to_numeric(df['기준_시간대'], errors='coerce').fillna(0).astype(int)
        df['시'] = (df['기준_시간대'] // 100).astype(int)
        df['분'] = (df['기준_시간대'] % 100).astype(int)

    start_df = df[df['집계_기준'].astype(str).str.contains('출발', na=False)]
    end_df   = df[df['집계_기준'].astype(str).str.contains('도착', na=False)]

    start_cnt = (start_df.groupby(['기준_날짜','시','시작_대여소_ID'])['전체_건수']
                         .sum().reset_index(name='대여건수'))
    end_cnt = (end_df.groupby(['기준_날짜','시','종료_대여소_ID'])['전체_건수']
                       .sum().reset_index(name='반납건수')
                       .rename(columns={'종료_대여소_ID':'시작_대여소_ID'}))

    flow = pd.merge(start_cnt, end_cnt,
                    on=['기준_날짜','시','시작_대여소_ID'], how='outer').fillna(0)
    flow['순유출입'] = flow['대여건수'] - flow['반납건수']

    flow = flow.merge(stations_df[['시작_대여소_ID','capacity']],
                      on='시작_대여소_ID', how='left')
    flow['capacity'] = flow['capacity'].fillna(0)

    flow['risk'] = flow['순유출입'] / flow['capacity'].replace(0, np.nan)
    flow['risk'] = flow['risk'].fillna(0)
    flow['가능'] = (flow['risk'] < thresh).astype(int)

    return flow

# test flow 생성
test_raw = pd.read_parquet('../../Data/2025_data.parquet')
flow_test = build_flow(test_raw, df1, thresh=THRESH)

# feature_cols = ['capacity','순유출입']
feature_cols = ['capacity']
X_test = flow_test[feature_cols]
y_test = flow_test['가능']
flow_test.head()


,기준_날짜,시,시작_대여소_ID,대여건수,반납건수,순유출입,capacity,risk,가능
0,20250109,0,ST-1029,0.0,2.0,-2.0,10.0,-0.200000,1
1,20250109,0,ST-1032,1.0,1.0,0.0,26.0,0.000000,1
2,20250109,0,ST-1038,1.0,0.0,1.0,10.0,0.100000,1
3,20250109,0,ST-1331,1.0,0.0,1.0,9.0,0.111111,1
4,20250109,0,ST-1481,1.0,0.0,1.0,15.0,0.066667,1


In [37]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# 피처/타깃
target = '가능'
# feature_cols = [c for c in flow.columns if 'lag' in c]
feature_cols = ['capacity','순유출입']

# print(feature_cols)

x_train = flow[feature_cols]
y_train = flow[target]

# 모델
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

# 평가
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]

cv_scores = []
for x in range(10):
    scores = cross_val_score(
        LogisticRegression(),
        x_train,
        y_train,
        n_jobs=-1,
        cv=5,
        scoring='accuracy'
    )
    cv_scores.append(scores.mean())
    # print(cv_scores.mean())
print(cv_scores) 

print("Accuracy:", accuracy_score(y_test, pred))
print("F1:", f1_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

# 평가
# pred = reg.predict(X_test)
mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- 순유출입


In [ ]:
# import numpy as np
# from sklearn.model_selection import train_test_split, cross_val_score
# from lightgbm import LGBMClassifier
# from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# target = '가능'
# feature_cols = ['capacity', 'risk', '순유출입']

# flow = flow.sort_values(['기준_날짜', '시'])

# X = flow[feature_cols]
# y = flow[target]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, random_state=42, test_size=0.3
# )

# clf = LGBMClassifier(
#     n_estimators=300,
#     learning_rate=0.05,
#     max_depth=-1,
#     num_leaves=31,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42
# )

# clf.fit(X_train, y_train)

# pred = clf.predict(X_test)
# prob = clf.predict_proba(X_test)[:, 1]

# scores = cross_val_score(
#     LGBMClassifier(random_state=42),
#     X_train,
#     y_train,
#     n_jobs=-1,
#     cv=5,
#     scoring='accuracy'
# )

# print("CV Accuracy:", scores.mean())
# print("Accuracy:", accuracy_score(y_test, pred))
# print("F1:", f1_score(y_test, pred))
# print("ROC-AUC:", roc_auc_score(y_test, prob))

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

target = '가능'
# feature_cols = ['capacity', 'risk', '순유출입']

# feature_cols = ['capacity','순유출입']
feature_cols = ['capacity']
# print(feature_cols)

x_train = flow[feature_cols]
y_train = flow[target]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, random_state=42, test_size=0.3
# )

reg = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

reg.fit(x_train, y_train)
pred = reg.predict(X_test)

scores = cross_val_score(
    LGBMRegressor(random_state=42),
    x_train,
    y_train,
    n_jobs=-1,
    cv=5,
    scoring='r2'
)

mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("CV R2:", scores.mean())
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000321 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3
[LightGBM] [Info] Number of data points in the train set: 19255, number of used features: 1
[LightGBM] [Info] Start training from score 0.862633
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

In [40]:
flow[['가능','capacity','risk','순유출입']].corr()

,가능,capacity,risk,순유출입
가능,1.000000,0.111369,-0.639632,-0.606493
capacity,0.111369,1.000000,-0.059473,-0.058044
risk,-0.639632,-0.059473,1.000000,0.973343
순유출입,-0.606493,-0.058044,0.973343,1.000000


In [ ]:
flow[['가능','capacity','risk','순유출입']].head(20)

In [ ]:
# 누출 제거 버전: risk 제외하고 재평가
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

feature_cols_noleak = ['capacity', '순유출입']

X_train_noleak = flow[feature_cols_noleak]
y_train_noleak = flow['가능']

# test는 앞에서 만든 flow_test 사용
X_test_noleak = flow_test[feature_cols_noleak]
y_test_noleak = flow_test['가능']

clf_noleak = LGBMClassifier(random_state=42)
clf_noleak.fit(X_train_noleak, y_train_noleak)

pred_noleak = clf_noleak.predict(X_test_noleak)
prob_noleak = clf_noleak.predict_proba(X_test_noleak)[:, 1]

print('Accuracy:', accuracy_score(y_test_noleak, pred_noleak))
print('F1:', f1_score(y_test_noleak, pred_noleak))
print('ROC-AUC:', roc_auc_score(y_test_noleak, prob_noleak))


[LightGBM] [Info] Number of positive: 16610, number of negative: 2645
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000291 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 32
[LightGBM] [Info] Number of data points in the train set: 19255, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.862633 -> initscore=1.837334
[LightGBM] [Info] Start training from score 1.837334
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in